In [ ]:
# Example: Advanced Indexing Strategies

indexing_examples = """
-- 1. B-tree Index (Default, best for equality and range queries)
CREATE INDEX idx_employees_salary ON employees (salary);
CREATE INDEX idx_orders_date ON orders (order_date DESC);

-- 2. Hash Index (Good for equality only, not range queries)
CREATE INDEX idx_users_email_hash ON users USING HASH (email);

-- 3. GiST (Generalized Search Tree - for complex data types)
CREATE INDEX idx_locations_gist ON locations 
USING GiST (coordinates);

-- 4. GIN (Generalized Inverted Index - for full-text search and arrays)
CREATE INDEX idx_documents_fts ON documents 
USING GIN (search_vector);

-- Array containment search
CREATE INDEX idx_tags_gin ON articles 
USING GIN (tags);

-- 5. BRIN (Block Range Index - large tables with sorted data)
CREATE INDEX idx_logs_date_brin ON logs (created_at) 
USING BRIN;

-- Multi-column (Composite) Indexes
CREATE INDEX idx_orders_customer_date ON orders (customer_id, order_date DESC);

-- Partial Index (index only matching rows)
CREATE INDEX idx_active_users ON users (email) 
WHERE status = 'active';

-- Expression Index (index on computed values)
CREATE INDEX idx_lower_email ON users (LOWER(email));

-- Covering Index (INCLUDE clause - PostgreSQL 11+)
CREATE INDEX idx_products_search ON products (name) 
INCLUDE (price, stock);

-- Index Statistics and Maintenance
-- Analyze index usage
SELECT 
    schemaname,
    tablename,
    indexname,
    idx_scan,
    idx_tup_read,
    idx_tup_fetch
FROM pg_stat_user_indexes
ORDER BY idx_scan DESC;

-- Find unused indexes
SELECT 
    schemaname,
    tablename,
    indexname,
    idx_scan
FROM pg_stat_user_indexes
WHERE idx_scan = 0 
AND indexrelname NOT LIKE 'pg_toast%'
ORDER BY pg_relation_size(indexrelid) DESC;

-- Rebuild/Reindex (remove bloat)
REINDEX INDEX idx_employees_salary;
REINDEX TABLE employees;

-- Check index size
SELECT 
    indexrelname,
    pg_size_pretty(pg_relation_size(indexrelid)) as size
FROM pg_stat_user_indexes
ORDER BY pg_relation_size(indexrelid) DESC;

-- Disable/Enable indexes for testing
ALTER INDEX idx_employees_salary UNUSABLE;  -- Disable
ALTER INDEX idx_employees_salary REBUILD;   -- Rebuild
"""

print("Advanced Indexing Examples:")
print(indexing_examples)

# Function to analyze index performance
def analyze_index_performance(conn):
    '''Show index usage statistics'''
    if conn is None:
        print("No database connection")
        return
    
    try:
        query = """
        SELECT 
            tbl.tablename,
            idx.indexname,
            idx.idx_scan as scans,
            idx.idx_tup_read as tuples_read,
            idx.idx_tup_fetch as tuples_fetched,
            pg_size_pretty(pg_relation_size(idx.indexrelid)) as size
        FROM pg_stat_user_indexes idx
        JOIN pg_tables tbl ON idx.relname = tbl.tablename
        ORDER BY idx.idx_scan DESC;
        """
        
        cursor = conn.cursor(cursor_factory=RealDictCursor)
        cursor.execute(query)
        
        print("\\nIndex Performance Statistics:")
        for row in cursor.fetchall():
            print(f"Table: {row['tablename']}, Index: {row['indexname']}")
            print(f"  Scans: {row['scans']}, Tuples Read: {row['tuples_read']}, Size: {row['size']}")
        
        cursor.close()
    except Exception as e:
        print(f"Error analyzing indexes: {e}")

print("\\nFunction 'analyze_index_performance()' is available for use")

# Summary and best practices
summary = """
INDEX OPTIMIZATION BEST PRACTICES:
1. Create indexes on columns used in WHERE, JOIN, and ORDER BY clauses
2. Use composite indexes for multi-column queries
3. Prefer B-tree for most general-purpose queries
4. Use GIN for full-text search and array operations
5. Use partial indexes to reduce size of frequently searched subsets
6. Profile with EXPLAIN ANALYZE before and after indexing
7. Monitor index usage with pg_stat_user_indexes
8. Regularly VACUUM and ANALYZE to maintain statistics
9. Remove unused indexes to reduce INSERT/UPDATE/DELETE overhead
10. Consider index bloat and run REINDEX periodically
"""

print(summary)

## 8. Advanced Indexing Strategies

Proper indexing is crucial for query performance. PostgreSQL supports various index types optimized for different use cases.

In [ ]:
# Example: Transactions and Isolation Levels

transactions_examples = """
-- PostgreSQL Isolation Levels (from least to most restrictive):

-- 1. READ UNCOMMITTED (PostgreSQL treats as READ COMMITTED)
-- Allows reading uncommitted changes from other transactions
BEGIN TRANSACTION ISOLATION LEVEL READ UNCOMMITTED;
SELECT * FROM accounts WHERE id = 1;
COMMIT;

-- 2. READ COMMITTED (DEFAULT)
-- Only reads committed data, prevents dirty reads
BEGIN TRANSACTION ISOLATION LEVEL READ COMMITTED;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;

-- 3. REPEATABLE READ
-- Prevents dirty reads and non-repeatable reads
BEGIN TRANSACTION ISOLATION LEVEL REPEATABLE READ;
SELECT balance FROM accounts WHERE id = 1;  -- At this level, value is frozen
-- Other transactions can modify the row, but this transaction won't see it
SELECT balance FROM accounts WHERE id = 1;  -- Same value
COMMIT;

-- 4. SERIALIZABLE
-- Highest isolation level, treats concurrent transactions as if serialized
BEGIN TRANSACTION ISOLATION LEVEL SERIALIZABLE;
SELECT COUNT(*) FROM orders;
INSERT INTO orders (customer_id, amount) VALUES (1, 99.99);
COMMIT;

-- Basic transaction with rollback
BEGIN;
UPDATE accounts SET balance = balance - 1000 WHERE id = 1;
UPDATE accounts SET balance = balance + 1000 WHERE id = 2;
-- If everything is OK, commit
COMMIT;
-- Or rollback if error detected
ROLLBACK;

-- Savepoints for partial rollback
BEGIN;
UPDATE products SET stock = stock - 5 WHERE id = 1;
SAVEPOINT before_second_update;
UPDATE products SET stock = stock - 3 WHERE id = 2;
ROLLBACK TO SAVEPOINT before_second_update;  -- Undo only the second update
COMMIT;

-- Detecting lock deadlocks and timeouts
SET deadlock_timeout = '5s';
"""

print("Transaction and Isolation Level Examples:")
print(transactions_examples)

# Function to execute transaction
def execute_transaction(conn, statements, isolation_level='READ COMMITTED'):
    '''Execute transaction with specified isolation level'''
    if conn is None:
        print("No database connection")
        return
    
    try:
        cursor = conn.cursor()
        cursor.execute(f'SET TRANSACTION ISOLATION LEVEL {isolation_level}')
        
        for statement in statements:
            cursor.execute(statement)
        
        conn.commit()
        print(f"✓ Transaction committed successfully at {isolation_level} level")
        cursor.close()
    except Exception as e:
        conn.rollback()
        print(f"✗ Transaction rolled back due to error: {e}")

print("\\nFunction 'execute_transaction()' is available for use")

## 7. Transactions and Isolation Levels

PostgreSQL supports ACID properties and multiple isolation levels to handle concurrent access. Understand the trade-offs between consistency and performance.

In [ ]:
# Example: Query Optimization with EXPLAIN ANALYZE

optimization_examples = """
-- EXPLAIN without execution
EXPLAIN
SELECT e.name, e.salary, d.department_name
FROM employees e
JOIN departments d ON e.department_id = d.id
WHERE e.salary > 50000;

-- EXPLAIN ANALYZE (actually executes the query)
EXPLAIN ANALYZE
SELECT e.name, e.salary, d.department_name
FROM employees e
JOIN departments d ON e.department_id = d.id
WHERE e.salary > 50000;

-- EXPLAIN with more details
EXPLAIN (ANALYZE, VERBOSE, BUFFERS)
SELECT COUNT(*)
FROM orders o
JOIN order_items oi ON o.id = oi.order_id
WHERE o.order_date > '2024-01-01';

-- Understanding output key metrics:
-- Seq Scan vs Index Scan: Sequential full table vs index usage
-- Rows: Estimated vs Actual (differences indicate stale statistics)
-- Buffers: Cache hits indicate efficiency

-- Query that might benefit from indexing
EXPLAIN ANALYZE
SELECT * FROM customers
WHERE email LIKE '%@gmail.com%'
AND created_at > '2024-01-01'
ORDER BY last_purchase DESC;

-- After creating appropriate index
CREATE INDEX idx_customers_email_date ON customers (email, created_at);

-- Rerun EXPLAIN to see improvement
EXPLAIN ANALYZE
SELECT * FROM customers
WHERE email LIKE '%@gmail.com%'
AND created_at > '2024-01-01'
ORDER BY last_purchase DESC;

-- Analyze for slow queries
-- Enable query logging
SET log_min_duration_statement = 1000;  -- Log queries > 1000ms

-- Check autovacuum status and statistics
SELECT schemaname, tablename, last_vacuum, last_analyze
FROM pg_stat_user_tables
WHERE last_analyze IS NULL OR last_analyze < now() - interval '1 week';
"""

print("Query Optimization Examples:")
print(optimization_examples)

# Function to analyze query performance
def analyze_query_performance(conn, query):
    '''Execute EXPLAIN ANALYZE and display results'''
    if conn is None:
        print("No database connection")
        return
    
    try:
        cursor = conn.cursor()
        explain_query = f"EXPLAIN ANALYZE {query}"
        cursor.execute(explain_query)
        results = cursor.fetchall()
        print("\\nQuery Execution Plan:")
        for row in results:
            print(row[0])
        cursor.close()
    except Exception as e:
        print(f"Error analyzing query: {e}")

print("\\nFunction 'analyze_query_performance()' is available for use")

## 6. Query Optimization and EXPLAIN ANALYZE

Understanding query execution plans is crucial for optimization. `EXPLAIN ANALYZE` shows how PostgreSQL executes your queries.

In [ ]:
# Example: Full-Text Search

fts_examples = """
-- Create table with full-text search support
CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    title TEXT,
    content TEXT,
    search_vector tsvector GENERATED ALWAYS AS (
        to_tsvector('english', coalesce(title, '') || ' ' || coalesce(content, ''))
    ) STORED
);

-- Create GIN index for fast full-text searches
CREATE INDEX idx_search_vector ON documents USING GIN (search_vector);

-- Insert sample data
INSERT INTO documents (title, content) VALUES
('PostgreSQL Basics', 'Learn the fundamentals of PostgreSQL database'),
('Advanced SQL', 'Complex queries and optimization techniques'),
('Full-Text Search Guide', 'How to implement powerful search functionality');

-- Basic full-text search with @@ operator
SELECT 
    id,
    title,
    content,
    ts_rank(search_vector, query) as rank
FROM documents,
plainto_tsquery('english', 'PostgreSQL search') query
WHERE search_vector @@ query
ORDER BY rank DESC;

-- Search with multiple terms (AND)
SELECT * FROM documents
WHERE search_vector @@ to_tsquery('english', 'PostgreSQL & search');

-- Search with OR
SELECT * FROM documents
WHERE search_vector @@ to_tsquery('english', 'database | search');

-- Search with phrase
SELECT * FROM documents
WHERE search_vector @@ phraseto_tsquery('english', 'full text search');

-- Advanced ranking with weights
SELECT 
    id,
    title,
    ts_headline('english', content, query, 'StartSel=<mark>, StopSel=</mark>') as headline,
    ts_rank_cd(search_vector, query, 32) as rank
FROM documents,
plainto_tsquery('english', 'PostgreSQL') query
WHERE search_vector @@ query
ORDER BY rank DESC;
"""

print("Full-Text Search Examples:")
print(fts_examples)

## 5. Full-Text Search

PostgreSQL provides robust full-text search capabilities using `tsvector` (searchable text) and `tsquery` (search queries). This is more efficient than LIKE for text search.

In [ ]:
# Example: JSON and JSONB Operations

json_examples = """
-- JSON Extraction Operators
-- -> returns JSON, ->> returns TEXT

-- Store and query JSON data
CREATE TABLE api_responses (
    id SERIAL PRIMARY KEY,
    response JSONB
);

INSERT INTO api_responses (response) VALUES
('{"user": {"id": 1, "name": "John", "email": "john@example.com"}, "status": "active"}'),
('{"user": {"id": 2, "name": "Jane", "email": "jane@example.com"}, "status": "inactive"}');

-- Extract JSON values
SELECT 
    id,
    response -> 'user' ->> 'name' as name,
    response -> 'user' ->> 'email' as email,
    response ->> 'status' as status
FROM api_responses;

-- Extract nested arrays
SELECT 
    id,
    jsonb_array_elements(response -> 'tags') as tag
FROM articles
WHERE response @> '{"category": "tutorial"}'::jsonb;

-- JSONB Operators
-- @> contains
-- <@ contained by
-- ? has key
-- || concatenate
-- - delete

-- Check if JSONB contains a value
SELECT * FROM api_responses 
WHERE response @> '{"status": "active"}'::jsonb;

-- Update JSONB values
UPDATE api_responses 
SET response = jsonb_set(response, '{user,status}', '"premium"'::jsonb)
WHERE response -> 'user' ->> 'id' = '1';

-- Convert JSONB to rows and columns
SELECT 
    key,
    value
FROM api_responses,
LATERAL jsonb_each(response -> 'user')
WHERE id = 1;
"""

print("JSON and JSONB Examples:")
print(json_examples)

## 4. JSON and JSONB Operations

PostgreSQL provides rich JSON support with operators for extraction, manipulation, and querying. JSONB is the preferred type as it's parsed and indexed.

In [ ]:
# Example: CTEs (WITH clauses) - Simple and Recursive

cte_examples = """
-- Simple CTE: Calculate average salary by department
WITH dept_salary_stats AS (
    SELECT 
        department_id,
        AVG(salary) as avg_salary,
        COUNT(*) as employee_count
    FROM employees
    GROUP BY department_id
)
SELECT 
    e.employee_id,
    e.salary,
    d.avg_salary,
    e.salary - d.avg_salary as salary_diff
FROM employees e
JOIN dept_salary_stats d ON e.department_id = d.department_id;

-- Multiple CTEs
WITH sales_summary AS (
    SELECT 
        customer_id,
        SUM(amount) as total_sales
    FROM orders
    GROUP BY customer_id
),
customer_ranking AS (
    SELECT 
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) as rank
    FROM sales_summary
)
SELECT * FROM customer_ranking WHERE rank <= 10;

-- Recursive CTE: Generate organizational hierarchy
WITH RECURSIVE employee_hierarchy AS (
    -- Base case: top-level managers
    SELECT 
        employee_id,
        name,
        manager_id,
        0 as level
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive case: subordinates
    SELECT 
        e.employee_id,
        e.name,
        e.manager_id,
        eh.level + 1
    FROM employees e
    JOIN employee_hierarchy eh ON e.manager_id = eh.employee_id
    WHERE eh.level < 5  -- Prevent infinite recursion
)
SELECT * FROM employee_hierarchy ORDER BY level, employee_id;
"""

print("CTE Examples:")
print(cte_examples)

## 3. Common Table Expressions (CTEs)

CTEs (WITH clauses) allow you to create temporary result sets that simplify complex queries and enable recursive operations.

In [ ]:
# Example: Window Functions with ROW_NUMBER, RANK, DENSE_RANK
window_functions_example = """
-- ROW_NUMBER: Unique sequential number for each row
SELECT 
    employee_id,
    salary,
    department_id,
    ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC) as row_num,
    RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) as rank,
    DENSE_RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) as dense_rank
FROM employees
ORDER BY department_id, salary DESC;

-- LAG and LEAD: Compare with previous/next rows
SELECT 
    order_date,
    order_amount,
    LAG(order_amount) OVER (ORDER BY order_date) as prev_order,
    LEAD(order_amount) OVER (ORDER BY order_date) as next_order,
    order_amount - LAG(order_amount) OVER (ORDER BY order_date) as difference
FROM orders
ORDER BY order_date;

-- Cumulative sum using SUM() window function
SELECT 
    month,
    sales,
    SUM(sales) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as cumulative_sales,
    AVG(sales) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as moving_avg_3month
FROM monthly_sales
ORDER BY month;
"""

print("Window Functions Examples:")
print(window_functions_example)

## 2. Window Functions and Analytical Queries

Window functions perform calculations across a set of rows related to the current row. They include:
- **ROW_NUMBER()**: Assigns unique numbers to rows
- **RANK()**: Assigns ranks with gaps on ties
- **DENSE_RANK()**: Assigns ranks without gaps
- **LAG() / LEAD()**: Access previous/next rows
- **PERCENT_RANK()**: Percentage rank of a row

In [ ]:
# Install required packages
# Uncomment if needed: !pip install psycopg2-binary sqlalchemy

import psycopg2
from psycopg2 import sql
from psycopg2.extras import RealDictCursor, execute_values
import pandas as pd
from datetime import datetime
import json

print("Libraries imported successfully!")

# Connection parameters - CONFIGURE THESE FOR YOUR DATABASE
conn_params = {
    'host': 'localhost',
    'port': 5432,
    'database': 'postgres',
    'user': 'postgres',
    'password': 'your_password'
}

# Function to establish connection
def get_connection():
    try:
        conn = psycopg2.connect(**conn_params)
        print("✓ Connected to PostgreSQL successfully!")
        return conn
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        return None

# Test connection
conn = get_connection()
if conn:
    conn.close()
    print("Connection test completed")

## 1. Import Required Libraries and Connect to PostgreSQL

We'll use `psycopg2` for direct PostgreSQL connections and demonstrate connection management.

# Advanced PostgreSQL Learning Notebook

This comprehensive notebook covers advanced PostgreSQL concepts and techniques for database optimization and complex querying.